In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display, Markdown
import numpy as np
import sqlite3, json
import pandas as pd
import duckdb
import time

In [2]:
def compress_ship_data_duckdb_further(db_path, table_name, min_lon=125.678, max_lon=131.229, max_lat=36.001):
    db_path = db_path.replace("\\", "/") 
    con = duckdb.connect(database=':memory:')
    con.execute("INSTALL sqlite; LOAD sqlite;")
    con.execute(f"ATTACH '{db_path}' AS sqlite_db (TYPE SQLITE);")

    query = f"""
    -- [STEP 1] Raw 데이터 로드 및 1차 트리거 (정수 변환 비교로 정밀도 확보)
    WITH raw_data AS (
        SELECT *,
            CAST(timestamp AS TIMESTAMP) as ts,
            CAST(longitude AS DOUBLE) as lon_val,
            CAST(latitude AS DOUBLE) as lat_val,
            CAST(course AS DOUBLE) as c_course,
            CAST(speed AS DOUBLE) as s_speed,
            row_number() OVER () as temp_row_idx
        FROM sqlite_db.{table_name}
        WHERE longitude >= {min_lon} AND longitude <= {max_lon} AND latitude <= {max_lat}
    ),
    ordered_data AS (
        SELECT *,
            LAG(lon_val) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_lon,
            LAG(lat_val) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_lat,
            LAG(c_course) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_course,
            LAG(s_speed) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_speed
        FROM raw_data
    ),
    diff_calc AS (
        SELECT *,
            CASE WHEN CAST(lon_val * 1000 AS BIGINT) != CAST(COALESCE(prev_lon, lon_val) * 1000 AS BIGINT) 
                   OR CAST(lat_val * 1000 AS BIGINT) != CAST(COALESCE(prev_lat, lat_val) * 1000 AS BIGINT) 
                 THEN 1 ELSE 0 END as pos_change,
            CASE 
                WHEN (c_course - COALESCE(prev_course, c_course)) > 180 THEN (c_course - COALESCE(prev_course, c_course)) - 360
                WHEN (c_course - COALESCE(prev_course, c_course)) < -180 THEN (c_course - COALESCE(prev_course, c_course)) + 360
                ELSE (c_course - COALESCE(prev_course, c_course))
            END as course_diff,
            (s_speed - COALESCE(prev_speed, s_speed)) as speed_diff
        FROM ordered_data
    ),
    event_logic AS (
        SELECT *,
            -- 경계값 오차 방지를 위해 0.000001 보정 (Pandas와의 일치성 향상)
            CASE WHEN pos_change = 1 OR (ABS(course_diff) >= 19.999999 AND s_speed >= 0.999999) OR ABS(speed_diff) >= 1.999999 THEN 1 ELSE 0 END as event_trigger
        FROM diff_calc
    ),
    grouping_v1 AS (
        SELECT *,
            SUM(event_trigger) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as group_id_v1
        FROM event_logic
    ),
    summarized_v1 AS (
        -- [1차 압축] ANY_VALUE 대신 FIRST를 사용하여 Pandas .first()와 100% 일치화
        SELECT 
            ShipName, group_id_v1,
            FIRST(mmsi) as mmsi, FIRST(higher_types) as higher_types, FIRST(radius) as radius,
            MIN(ts) as start_time, MAX(ts) as end_time,
            FIRST(lon_val) as lon, FIRST(lat_val) as lat,
            FIRST(c_course) as first_course, AVG(s_speed) as avg_speed,
            FIRST(course_diff) as turn_val, FIRST(speed_diff) as accel_val
        FROM grouping_v1 
        GROUP BY ShipName, group_id_v1
    ),
    -- [STEP 2] 상태 판별 (부동소수점 오차 차단)
    status_calc AS (
        SELECT *,
            avg_speed - COALESCE(LAG(avg_speed) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1), avg_speed) as group_speed_diff
        FROM summarized_v1
    ),
    status_final AS (
        SELECT *,
            CASE 
                -- 1.0, 20.0 등의 경계값을 소수점 8자리에서 반올림 후 비교하여 Pandas와 일치시킴
                WHEN ROUND(avg_speed, 8) < 1.0 THEN '정박/대기'
                ELSE TRIM(CONCAT_WS(' ',
                    CASE WHEN ROUND(ABS(turn_val), 8) >= 20.0 AND ROUND(avg_speed, 8) >= 1.0 
                         THEN (CASE WHEN turn_val > 0 THEN '우선회' ELSE '좌선회' END) || '(' || ROUND(ABS(turn_val), 1) || '°)' ELSE '' END,
                    CASE WHEN ROUND(group_speed_diff, 8) >= 2.0 THEN '가속' 
                         WHEN ROUND(group_speed_diff, 8) < -2.0 THEN '감속' ELSE '' END,
                    CASE WHEN ROUND(avg_speed, 8) >= 5.0 THEN '이동/통과' ELSE '저속 운항' END
                ))
            END as status
        FROM status_calc
    ),
    -- [STEP 3] 2차 압축
    v2_trigger AS (
        SELECT *,
            CASE WHEN status != COALESCE(LAG(status) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1), status) 
                 THEN 1 ELSE 0 END as status_change
        FROM status_final
    ),
    v2_grouping AS (
        SELECT *,
            SUM(status_change) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as group_id
        FROM v2_trigger
    )
    -- [STEP 4] 최종 요약 (집계 방식 일치)
    SELECT 
        ShipName, group_id,
        FIRST(mmsi) as mmsi, FIRST(higher_types) as higher_types, FIRST(radius) as radius,
        MIN(start_time) as start_time, MAX(end_time) as end_time,
        FIRST(lon) as lon, FIRST(lat) as lat,
        FIRST(first_course) as first_course,
        AVG(avg_speed) as avg_speed,
        FIRST(turn_val) as turn_val,
        FIRST(accel_val) as accel_val,
        FIRST(status) as status
    FROM v2_grouping
    GROUP BY ShipName, group_id
    ORDER BY ShipName, start_time
    """

    df_result = con.execute(query).df()
    con.execute("DETACH sqlite_db;")
    return df_result


In [3]:
start = time.time()
testdb = "D:/AI_team/github/Vision_AI_RnD_team/projects/test_project/ais_weather/test.db"
res_2 = compress_ship_data_duckdb_further(testdb, "AIS_category")
print(time.time() - start)

1.24237060546875


In [5]:
res_2

,ShipName,group_id,mmsi,higher_types,radius,start_time,end_time,lon,lat,first_course,avg_speed,turn_val,accel_val,status
0,AGS- 신천지,0.0,352001086,others,5000,2022-12-01 00:00:14,2022-12-01 14:10:12,128.793800,35.077283,140.0,0.008718,0.0,0.0,정박/대기
1,AGS- 신천지,1.0,352001086,others,5000,2022-12-01 14:15:31,2022-12-01 14:15:31,128.791850,35.076550,248.0,2.600000,62.0,1.9,우선회(62.0°) 가속 저속 운항
2,AGS- 신천지,2.0,352001086,others,5000,2022-12-01 14:18:13,2022-12-01 14:18:13,128.791183,35.076317,246.0,3.300000,-2.0,0.7,저속 운항
3,AGS- 신천지,3.0,352001086,others,5000,2022-12-01 14:20:01,2022-12-01 14:24:12,128.786750,35.071650,205.0,7.200000,-41.0,3.9,좌선회(41.0°) 가속 이동/통과
4,AGS- 신천지,4.0,352001086,others,5000,2022-12-01 14:25:11,2022-12-01 14:25:11,128.781283,35.058417,195.0,11.200000,-10.0,4.0,가속 이동/통과
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4388,SSM-052,33.0,563872000,others,5000,2022-12-14 15:26:11,2022-12-14 15:45:00,127.836058,34.716018,127.7,12.685714,-5.1,1.3,이동/통과
4389,SSM-052,34.0,563872000,others,5000,2022-12-14 15:51:40,2022-12-14 15:51:40,127.920303,34.662048,170.2,12.400000,48.0,-0.7,우선회(48.0°) 이동/통과
4390,SSM-052,35.0,563872000,others,5000,2022-12-14 15:55:41,2022-12-14 16:14:38,127.923287,34.647813,170.1,13.040000,-0.1,0.6,이동/통과
4391,SSM-052,36.0,563872000,others,5000,2022-12-14 16:16:40,2022-12-14 16:16:40,127.940357,34.575545,170.1,9.600000,-0.4,-3.3,감속 이동/통과


In [6]:
mmsi_counts = res_2['mmsi'].value_counts()
print(mmsi_counts[:20])

mmsi
538008133    739
305062000    673
441963000    253
431016257    233
371407000    214
371002000    141
352002106    139
372024000    108
374314000    103
440562000     92
431501855     88
273358000     82
538005766     80
259005000     71
311039000     70
440314210     68
563161700     66
431400833     63
351854000     60
352001086     53
Name: count, dtype: int64


In [8]:
# 1. 먼저 ShipName과 timestamp 기준으로 전체 데이터를 정렬합니다.
# (timestamp가 문자열이라면 정렬 전 pd.to_datetime으로 변환하는 것이 정확합니다)
res_2_sorted = res_2.sort_values(by=['ShipName', 'start_time'], ascending=True)

target_mmsi = [563161700, 352002106, 441963000, 431016257, 538008133, 305062000, 371407000, 372024000, 371002000, 374314000]
filtered_mmsi = res_2_sorted[res_2_sorted['mmsi'].isin(target_mmsi)].copy()

# 시간 필터링
start_time = "2022-12-01 00:00:00"
end_time = "2022-12-01 23:59:59"

time_filtered = filtered_mmsi[(filtered_mmsi['start_time'] >= start_time) & (filtered_mmsi['end_time'] <= end_time)].copy()
time_filtered

,ShipName,group_id,mmsi,higher_types,radius,start_time,end_time,lon,lat,first_course,avg_speed,turn_val,accel_val,status
53,AOE-59 화천,0.0,431016257,auxiliary,5500,2022-12-01 06:06:12,2022-12-01 08:20:52,129.950000,33.475000,0.0,0.400000,0.0,0.0,정박/대기
54,AOE-59 화천,1.0,431016257,auxiliary,5500,2022-12-01 08:32:19,2022-12-01 08:32:19,129.951193,33.482015,322.6,2.600000,-98.3,1.8,좌선회(98.3°) 저속 운항
55,AOE-59 화천,2.0,431016257,auxiliary,5500,2022-12-01 08:35:17,2022-12-01 14:50:08,129.950658,33.482428,157.9,0.169807,-164.7,-1.5,정박/대기
56,AOE-59 화천,3.0,431016257,auxiliary,5500,2022-12-01 14:55:11,2022-12-01 14:55:11,129.955567,33.489872,53.4,8.400000,88.4,7.6,우선회(88.4°) 가속 이동/통과
57,AOE-59 화천,4.0,431016257,auxiliary,5500,2022-12-01 15:07:08,2022-12-01 15:30:09,129.986445,33.505690,58.6,10.928571,5.2,1.5,이동/통과
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4102,SS-078 유관순,28.0,441963000,others,5000,2022-12-01 20:19:50,2022-12-01 20:19:50,126.551013,33.526870,188.1,4.800000,22.0,-11.3,우선회(22.0°) 감속 저속 운항
4103,SS-078 유관순,29.0,441963000,others,5000,2022-12-01 22:37:59,2022-12-01 22:37:59,126.767158,33.583043,84.7,20.800000,-103.4,16.0,좌선회(103.4°) 가속 이동/통과
4104,SS-078 유관순,30.0,441963000,others,5000,2022-12-01 22:53:08,2022-12-01 22:53:08,126.862718,33.558897,122.7,21.000000,38.0,0.2,우선회(38.0°) 이동/통과
4105,SS-078 유관순,31.0,441963000,others,5000,2022-12-01 23:08:18,2022-12-01 23:08:18,126.931473,33.496570,159.3,20.200000,36.6,-0.8,우선회(36.6°) 이동/통과


In [9]:
#2개 이상의 항적 각각 리스트로 묶음
ship_track_list = [
    group[1].to_dict(orient='records') 
    for group in time_filtered.groupby('ShipName', sort=False)
]

trajectory_json = json.dumps(ship_track_list, ensure_ascii=False, indent=4, default=str)
print(trajectory_json)

[
    [
        {
            "ShipName": "AOE-59 화천",
            "group_id": 0.0,
            "mmsi": 431016257,
            "higher_types": "auxiliary",
            "radius": 5500,
            "start_time": "2022-12-01 06:06:12",
            "end_time": "2022-12-01 08:20:52",
            "lon": 129.95,
            "lat": 33.475,
            "first_course": 0.0,
            "avg_speed": 0.4,
            "turn_val": 0.0,
            "accel_val": 0.0,
            "status": "정박/대기"
        },
        {
            "ShipName": "AOE-59 화천",
            "group_id": 1.0,
            "mmsi": 431016257,
            "higher_types": "auxiliary",
            "radius": 5500,
            "start_time": "2022-12-01 08:32:19",
            "end_time": "2022-12-01 08:32:19",
            "lon": 129.9511933,
            "lat": 33.482015,
            "first_course": 322.6,
            "avg_speed": 2.6,
            "turn_val": -98.29999999999995,
            "accel_val": 1.8,
            "status": "좌선회(98.

In [13]:
conn = sqlite3.connect('d:/AI_team/github/Vision_AI_RnD_team/projects/test_project/ais_weather/korea_weather.db', check_same_thread=False)
weather_cursor = conn.cursor()

# 4. 최종 DB 포맷 조립 (예: '2025-12-10 3')
current_hour_str = "2025-12-1 3"

query = """
    SELECT 
        b.지점명,
        b.latitude, 
        b.longitude, 
        w.*
    FROM 
        weather_buoy AS w
    JOIN 
        buoy_position AS b ON w.지점 = b.지점
    WHERE 
        w.일시 LIKE ?
"""

# 3. 쿼리 실행
search_param = f"{current_hour_str}:%"
weather_cursor.execute(query, (search_param,))
rows = weather_cursor.fetchall()

# 4. 데이터 출력 및 처리
if rows:
    # 2. Pandas 데이터프레임으로 로드
    col_names = [desc[0] for desc in weather_cursor.description]
    df = pd.DataFrame(rows, columns=col_names)

    target_cols = [0, 1, 2] + list(range(5, len(df.columns)))
    final_df = df.iloc[:, target_cols]
final_df

,지점명,latitude,longitude,풍속(m/s),풍향(deg),GUST풍속(m/s),현지기압(hPa),습도(%),기온(°C),수온(°C),최대파고(m),유의파고(m),평균파고(m),파주기(sec),파향(deg)
0,울릉도,37.4554,131.1144,7.0,319.0,9.2,1024.4,46,10.1,16.4,2.3,1.5,1.0,6.7,355
1,덕적도,37.2361,126.0188,3.7,219.0,4.8,1027.0,52,9.3,12.0,0.4,0.3,NaN,5.3,284
2,칠발도,34.7933,125.7769,3.3,104.0,4.1,1027.7,75,8.2,12.6,0.5,0.3,0.2,4.2,343
3,거문도,34.0014,127.5014,5.9,23.0,7.6,1027.1,61,9.5,18.2,0.9,0.5,0.4,3.3,357
4,거제도,34.7667,128.9000,5.5,325.0,9.1,1027.4,55,8.0,19.2,0.9,0.6,0.4,3.6,14
5,동해,37.5442,130.0000,5.7,338.0,7.8,1025.1,48,9.4,16.6,2.3,1.5,1.1,6.7,97
6,마라도,33.0833,126.0333,8.1,24.0,10.1,1026.3,60,12.5,NaN,1.1,0.8,0.5,4.2,14
7,외연도,36.2500,125.7500,NaN,NaN,NaN,1027.0,61,10.4,12.4,0.6,0.3,0.3,3.8,275
8,신안,34.7333,126.2417,3.6,54.0,3.6,1027.8,66,7.6,12.3,0.2,0.1,0.1,5.7,61
9,추자도,33.7936,126.1411,4.8,37.0,6.3,1027.4,61,10.0,15.7,0.7,0.4,0.3,2.9,12


In [18]:
#mmsi 두개 이상일 때 각각의 최근위치
last_rows_df = filtered_mmsi.groupby('mmsi').tail(1)
ref_coords = final_df[['latitude', 'longitude']].values
results = []

for _, ship_row in last_rows_df.iterrows():
    # 1. 선박 현재 위치 및 기본 정보 추출
    mmsi = ship_row['mmsi']
    current_pos = np.array([ship_row['lat'], ship_row['lon']])
    
    # 2. 거리 계산 및 가장 가까운 지점 인덱스 추출
    dist_sq = np.sum((ref_coords - current_pos)**2, axis=1)
    closest_idx = np.argmin(dist_sq)
    
    # 3. 가장 가까운 지점의 정보를 가져와서 선박 정보와 합치기
    closest_node_data = final_df.iloc[closest_idx].to_dict()
    results.append(closest_node_data)

weather_json = json.dumps(results, ensure_ascii=False, indent=4)
print(len(weather_json))
print(weather_json)

3879
[
    {
        "지점명": "울산",
        "latitude": 35.3453,
        "longitude": 129.8414,
        "풍속(m/s)": 5.9,
        "풍향(deg)": 337.0,
        "GUST풍속(m/s)": 8.2,
        "현지기압(hPa)": 1026.5,
        "습도(%)": 53,
        "기온(°C)": 8.8,
        "수온(°C)": 19.0,
        "최대파고(m)": 1.5,
        "유의파고(m)": 0.9,
        "평균파고(m)": 0.6,
        "파주기(sec)": 7.1,
        "파향(deg)": 23
    },
    {
        "지점명": "거제도",
        "latitude": 34.7667,
        "longitude": 128.9,
        "풍속(m/s)": 5.5,
        "풍향(deg)": 325.0,
        "GUST풍속(m/s)": 9.1,
        "현지기압(hPa)": 1027.4,
        "습도(%)": 55,
        "기온(°C)": 8.0,
        "수온(°C)": 19.2,
        "최대파고(m)": 0.9,
        "유의파고(m)": 0.6,
        "평균파고(m)": 0.4,
        "파주기(sec)": 3.6,
        "파향(deg)": 14
    },
    {
        "지점명": "소매물도",
        "latitude": 34.6214,
        "longitude": 128.538,
        "풍속(m/s)": 4.5,
        "풍향(deg)": 9.0,
        "GUST풍속(m/s)": 5.5,
        "현지기압(hPa)": 1027.6,
        "습도(%)": 51,
     

In [19]:
local_llm = ChatOpenAI(
            api_key="ai",
            model="openai/gpt-oss-20b",
            base_url="http://192.168.0.110:8000/v1",
        )

In [20]:
#vllm server alive check
res = local_llm.invoke("안녕")
res

AIMessage(content='안녕! 무엇을 도와드릴까요? 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 71, 'total_tokens': 149, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': None, 'id': 'chatcmpl-84e23af692534b39b6dc2b8a32244d69', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d5133-97ca-70b2-84b2-ef7f3b7b6020-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 71, 'output_tokens': 78, 'total_tokens': 149, 'input_token_details': {}, 'output_token_details': {}})

In [21]:
ais_prompt_template ="""
    # Role
    당신은 대한민국 주변 선박운행을 관제하는 베테랑 해상 관제사(VTS Operator)이자 선박 항적 분석 전문가 입니다. 
    제공된 요약된 항적 데이터(Summarized Trajectory)를 바탕으로 선박의 이동 패턴과 주요 이벤트를 전문적인 자연어로 묘사해주세요.
    
    #Input Data
    1. Summarized Trajectory
    {trajectory_data}
    2. Current Weather data
    {weather_data}
    
    # Guidelines
    1. 시간 순서대로 전체적인 항해 흐름을 요약해주세요.
    2. 요약할 때 기준은 "status"필드를 기준으로 하되 start_time과 end_time을 참고하세요. 
    3. trajectory_data는 ShipName이 같은 데이터끼리 리스트로 한번 더 묶여있으니 서로 ShipName이 서로 다른 배끼리 혼용하지 않도록 유의하세요.
    4. 첫번째 ShipName에 해당하는 현재날씨는 weather_data에서 첫번째 날씨 데이터를, 두번째 Shipnam에 해당하는 현재 날씨는 weather_data에서 두번째 날씨 데이터를 참고하세요.
    5. 전문적인 해상 관제 용어를 사용해도 되지만, 가독성이 좋게 작성하세요.
    6. 날씨데이터에 대해서 관제사가 고려하고 참고해야 할 사항에 대해 설명해주세요
    
    #Output Format
    1. 전체적인 항해 패턴을 간단하게 요약; 관제사가 관심을 가져야 할 특이 사항이 있을 시 간단하게 언급
    2. 현재 날씨에 대해 간단히 요약; 관제사가 관심을 가져야 할 날씨의 특이사항(풍향, 풍속 유의파고 등)이 있을 시 간단하게 언급 및 조치사항 설명
    3. 1, 2 항목을 종합적으로 정리해서 묘사
    """

summary_trajectory_template ="""
    #Input Data
    {all_analyses}
    
    # Guidelines
    1. Input Data의 정보를 종합해서 전체 배에 대한 내용을 종합적으로 요약해주세요
    2. 각각의 배에 특이사항을 중점적으로 요약하세요 
    3. 수치는 바뀌지 않도록 정확하게 참고해주세요
    4. 반복적인 내용은 짧게 요약하세요
    """

each_ship_prompt = ChatPromptTemplate.from_template(ais_prompt_template)
summary_trajectory_prompt = ChatPromptTemplate.from_template(summary_trajectory_template)

In [30]:
analyze_chain = each_ship_prompt | local_llm | StrOutputParser()

# --- 1. 개별 분석 함수 (스레드에서 실행될 단위) ---
def analyze_single_ship(data):
    # data는 {'idx': i, 'trajectory': t, 'weather': w} 형태
    res = analyze_chain.invoke({
        "trajectory_data": data['trajectory'],
        "weather_data": data['weather']
    })
    return f"[ship_{data['idx']} 분석 결과]\n{res}"

# --- 2. 병렬 처리를 수행하는 래퍼 함수 ---
def parallel_wrapper(inputs):
    # inputs: {'tracks': [...], 'weathers': [...]}
    tracks = inputs['tracks']
    weathers = inputs['weathers']
    
    tasks = [
        {'idx': i, 'trajectory': tracks[i], 'weather': weathers[i]} 
        for i in range(len(tracks))
    ]
    
    with ThreadPoolExecutor(max_workers=100) as executor:
        results_list = list(executor.map(analyze_single_ship, tasks))
    
    # 다음 체인(요약)을 위해 딕셔너리 형태로 반환
    return {
        "all_analyses": "\n\n".join(results_list),
        "ship_count": len(tracks)
    }

# --- 3. 하나의 거대한 체인으로 결합 ---
full_integrated_chain = (
    RunnableLambda(parallel_wrapper)  # 1단계: 병렬 분석 실행 및 결과 취합
    | summary_trajectory_prompt       # 2단계: 요약 프롬프트에 전달
    | local_llm                       # 3단계: 최종 요약 생성
    | StrOutputParser()               # 4단계: 텍스트만 추출
)

In [31]:
mark = []
async def astream_output():
    start_time = time.time()
    ttft = None
    # astream을 사용하여 비동기 스트리밍
    async for chunk in full_integrated_chain.astream({
        "tracks": ship_track_list,
        "weathers": results
    }):
        if ttft is None:
            ttft = time.time() - start_time
            print(f"{ttft}초")
        # 주피터 노트북에서 바로 출력 (end=""로 줄바꿈 없이 출력)
        print(chunk, end="", flush=True)
        mark.extend(chunk)

# 주피터에서 비동기 함수 실행
await astream_output()

50.11903238296509초
## 1. 전체 배에 대한 종합 요약  

| 배 | 주요 항해 패턴 | 날씨 상황 | 중점 비고 |
|---|---|---|---|
| **화천호 (ship_0)** | 06시 이후 12 kn정요속으로 30–90° 회전(좌·우 선회)  →  18시 이후 좁은 해역에서 지속적 좌·우 도선 | 풍향 337°, 풍속 5.9 m / s(≈12 kn), 파미 1.5 m | 제동·회전 토크가 높음. 12 kn 이하에서 비상속도 제한 권고. |
| **FF‑959 부산 (ship_1)** | 04–06, 13–18 시 장기간 정박·좌선회(≈30°) →  23시 10–11 kn 고속 | 풍향 325°, 풍속 5.5 m / s, 파 0.6 m(최고 0.9 m) | 정박 단계에서 선박이 끌리기 쉬우므로 속도·좌선회 관리 필요. |
| **FFG‑818 대구 (ship_2)** | 대부분 1–2 kn 좌선회·전진 →  14‑15 h 고속 30–50 kn & 25–40° 우선회 | 풍향 9°, 풍속 4.5 m / s, 파 0.3–0.5 m | 가속 → 전력 부하가 증가, 5‑10 초 파주기에서 전자기 간섭 주의. |
| **LST‑678 북한 (ship_3)** | 12 hr 연속 7–12 kn → 연속 56–68° 회전 → 18시 이후 목적지로 방향 변경 | 풍향 337°, 풍속 5.9 m / s, 파 0.6–1.5 m | 엔진에 고부하 발생. 전경 저항 증가로 토크 관리 필요. |
| **LST‑682 비로봉 (ship_4)** | 00–10 h 빠른 주행 12.7 kn → 10–12 h 정박·좌우 회전 → 15 h 가속·감속 | 풍향 4°, 풍속 4.4–6.0 m / s, 파 0.2–0.5 m | 저속·정박 중 바람 및 급류 영향 주시. |
| **LST‑683 향로봉 (ship_5)** | 00 시 가속 12–13 kn → 00–05 h 불가속·회전 → 05–06 h 급속 조정 | 풍향 25°, 풍속 6.4 m / s, 파 0.4–0.9 

In [29]:
display(Markdown(mark))


**전체 배 공동 요약**  
- **항해 속도 범위**: 대부분 0.1 ~ 22 kn 섭여, 12 ~ 15 kn이 가장 빈번.  
- **정박/대기 비율**: 3 시간 ~ 4 시간 이상 지속된 정박이 6곳에서 관측.  
- **주요 기상**  
  - 북서풍 5‑6 kn, 파고 0.4 ~ 1.5 m, 온도 8‑11 °C, 습도 50‑55 %.  
  - 변동이 최소이므로 대체로 “평온” 항해 환경.  
- **회전 패턴**  
  - 좌선회 10° ~ 90°, 우선회 20° ~ 100°가 주로 반복, 회전 수 10회 이상이 4개에서 관측.  
  - 18시 이후 회전이 흔하며, 곧 회전과 순간가속이 혼재.  
- **위치**  
  - 대부분 울산/거제도/소매물도 인근 해역에 주로 활동.  
- **공통 **조정 포인트**  
  - **정박 기간이 긴 경우(≥ 3 h)** → 연료, 엔진 부하, 피로 관리 필요.  
  - **18시 이후 회전 상승** → 보초·교통 통제 강화 필요.  
  - **무온도·파고 변동이 미미한** 지역에서 선박 전반의 회선이 안정적, 하지만 갑작스러운 진풍과 파고 변동 시 정밀 조종 소요.  

---

## 개별 배 특이사항

| 배명 | 주된 특이사항 | 핵심 포인트 |
|------|--------------|-------------|
| **ship_0** | 6시간 정박 후 급기동 → 14:55 ~ 15:30 급우선회 후 고속 주행 (10.9 kn). 18 시 이후 15회 초 이상의 회전. | 정박 시 엔진 점검, 18시 이후 회전 대비 보조 구조 필요. |
| **ship_1** | 12:30 ~ 18:00 동안 0.8‑2 kn 속도로 서쪽‑남‑서쪽 이동, 회전 20‑60°. 18시 이후 1‑2 kn에서 주황 회전. | 대형 회전 시 다른 선박과 간격 200 m 확보, 고속 이동이 드물어 피로 조정. |
| **ship_2** | 14:35 ~ 15:55 고속 통과 11‑12 kn, 가속률 4.7. 19 시 이후 다시 저속 1.3‑1.8 kn. | 14‑18 시 급가속, 19 시 이후 저속 복귀 주의. |
| **ship_3** | 04:45 ~ 04:48 급좌선회 56.2°, 05:00 ~ 05:12 다중 선회(5‑24°). 18:18 이후 장거리 직선이지만 7.8° 좌방향 선회 지속. | 04‑06 시 다중 선회: 물체 충돌 위험, 06‑18 시 연속 선회 감시 필요. |
| **ship_4** | 10:41 ~ 10:59 단기 우선회 40°, 이후 정박·대기 단계(11:30)과 16:29 평소 정박, 14‑15 시 급가속·감속(15 kn). | 10‑11 시/16 시 정박·대기 구간에서 간격·충돌 위험, 14‑15 시 급가속 위험성. |
| **ship_5** | 00:10 ~ 00:45 대규모 우선회(40‑45°) 복합, 01:54 ~ 03:30 직선 13‑15 kn, 05 시 짧은 정지·가속/감속 반복. | 대규모 회전 시 주변 선박과 거리를 주시, 오후 소규모 정지·가속 주의. |
| **ship_6** | 08:11 ~ 10:49 21 kn 가속 후 좌우 선회, 12:45 ~ 15:40 정박·대기(0.054 kn), 20:04 ~ 23:14 20 kn 이하 평균, 고속 선회>70°. | 02 시~05 시 21 kn 고속 및 산성, 20 시 이후 선회와 저속 혼재 주의. |

---

### 결론
- **정박·대기**가 4개 배에서 3–4 시간 지속되어 연료와 피로 관리가 중요.
- **18시 이후 회전**이 전반적으로 증가하여 보조 구조·교통 통제 필요.
- **고속(10 ~ 15 kn) 구간**에서 급가속, 큰 회전이 동반될 때 주변 선박과 거리 확보가 핵심.
- **기상**은 대부분 평온이지만, 북서풍으로 인한 바람 저항 및 짧은 진풍이 선박 동작에 가미될 수 있으므로 실시간 모니터링이 필요.
